# 🛰️ Notebook 01: Satellite Data Ingestion
**Space-Based Economic Intelligence Thesis**

This notebook demonstrates how to query and ingest satellite observations from Google Earth Engine (GEE) and Copernicus Data Space Ecosystem (CDSE) APIs.

### Environment Setup
We load libraries and initialize Earth Engine with our Google Cloud project ID.

In [ ]:
import os
import json
import ee
from dotenv import load_dotenv

load_dotenv()
project_id = os.getenv("GCP_PROJECT", "tesis-500804")

# Check for local client_secret.json to configure custom OAuth client
client_secret_path = os.path.abspath(os.path.join(os.getcwd(), "..", "client_secret.json"))
if not os.path.exists(client_secret_path):
    # Try current directory too just in case
    client_secret_path = os.path.abspath("client_secret.json")

if os.path.exists(client_secret_path):
    try:
        with open(client_secret_path, "r", encoding="utf-8") as f:
            secret_data = json.load(f)
        installed_info = secret_data.get("installed", {})
        if installed_info:
            client_id = installed_info.get("client_id")
            client_secret_val = installed_info.get("client_secret")
            if client_id and client_secret_val:
                print("Configuring Earth Engine custom client credentials from client_secret.json...")
                ee.oauth.CLIENT_ID = client_id
                ee.oauth.CLIENT_SECRET = client_secret_val
    except Exception as e:
        print(f"Failed to load client_secret.json: {e}")

try:
    ee.Initialize(project=project_id)
    print(f"Google Earth Engine initialized successfully with project: {project_id}!")
except Exception as e:
    print(f"Authentication/Initialization failed: {e}. Please run ee.Authenticate() in 00_setup.ipynb to log in.")

### Bounding Box Definition
Define the coordinates of the Salar de Atacama (Lithium case study) and Madre de Dios (Gold mining/Deforestation case study) study corridors.

In [ ]:
atacama_bbox = [-68.5, -23.8, -68.1, -23.2]
amazon_bbox = [-70.2, -13.1, -69.6, -12.6]

atacama_geometry = ee.Geometry.BBox(*atacama_bbox)
print("Study areas defined.")

### Querying Sentinel-2 Surface Reflectance
Query the Sentinel-2 Harmonized collection filtered by date range and cloud cover percent.

In [ ]:
s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(atacama_geometry)
    .filterDate("2024-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

best_image = s2_collection.first()
print("Best Sentinel-2 scene cloud cover:", best_image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo())